In [ ]:
# Install all required packages (run once in Google Colab)
!pip install transformers datasets seqeval torch numpy pandas matplotlib seaborn accelerate --quiet

In [ ]:
import sys
!"{sys.executable}" -m pip install datasets

In [ ]:
from datasets import load_dataset

dataset = load_dataset("conll2003")

print(dataset)
print(dataset["train"][0])

In [6]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

In [7]:
def tokenize_and_align_labels(example):
    tokenized_inputs = tokenizer(
        example["tokens"],
        truncation=True,
        is_split_into_words=True
    )
    
    labels = example["pos_tags"]   # use pos_tags OR chunk_tags

    word_ids = tokenized_inputs.word_ids()
    
    aligned_labels = []
    previous_word_idx = None

    for word_idx in word_ids:
        if word_idx is None:
            aligned_labels.append(-100)
        elif word_idx != previous_word_idx:
            aligned_labels.append(labels[word_idx])
        else:
            aligned_labels.append(-100)

        previous_word_idx = word_idx

    tokenized_inputs["labels"] = aligned_labels
    return tokenized_inputs

In [ ]:
tokenized_dataset = dataset.map(tokenize_and_align_labels, batched=False)

In [ ]:
from transformers import AutoModelForTokenClassification

num_labels = len(dataset["train"].features["pos_tags"].feature.names)

model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=num_labels
)

In [10]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01
)

In [4]:
import sys
print(sys.executable)

C:\Users\IT INFOTECH SOLUTION\AppData\Local\Programs\Python\Python310\python.exe


In [ ]:
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer, DataCollatorForTokenClassification
from datasets import load_dataset

# 1️⃣ Load dataset (replace with your own)
dataset = load_dataset("conll2003")  # example dataset

# 2️⃣ Load tokenizer and model
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=dataset["train"].features["ner_tags"].feature.num_classes
)

# 3️⃣ Align labels after tokenization
def align_labels_with_tokens(labels, word_ids):
    new_labels = []
    for i, word_id in enumerate(word_ids):
        if word_id is None:
            new_labels.append(-100)  # ignore token in loss
        else:
            new_labels.append(labels[word_id])
    return new_labels

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        is_split_into_words=True,
        padding="max_length",  # pad to max_length
        truncation=True,
        max_length=36
    )
    all_labels = []
    for i, labels in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        aligned_labels = align_labels_with_tokens(labels, word_ids)
        all_labels.append(aligned_labels)
    tokenized_inputs["labels"] = all_labels
    return tokenized_inputs

tokenized_dataset = dataset.map(tokenize_and_align_labels, batched=True)

# 4️⃣ Data collator for token classification
data_collator = DataCollatorForTokenClassification(tokenizer)

# 5️⃣ Training arguments (older version compatible)
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
)

# 6️⃣ Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator
)

# 7️⃣ Train
trainer.train()